# Unit 5: Evolving Your IPD Agent with Genetic Algorithms
### Completed student notebook — Elizabeth Ineintiemo

This notebook completes the genetic-algorithm TODOs, evolves both the original and improved agents, compares them in the full tournament, answers the analysis/reflection questions, and exports the final agent.


In [ ]:
import random
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from agents import Agent, INVEST, UNDERCUT
from game_engine import Game, Tournament

from agents import (
    AlwaysInvestAgent, AlwaysUndercutAgent, TitForTatAgent,
    GrimTriggerAgent, PavlovAgent, TitForTwoTatsAgent,
    GenerousTitForTatAgent, AdaptiveAgent, RandomAgent,
    SuspiciousTitForTatAgent, GradualAgent, HardMajorityAgent,
    SoftMajorityAgent, ProberAgent
)

random.seed(0)
print("✅ Imports complete!")


In [ ]:
# ============================================================================
# PART 1: THE EVOLVABLE AGENT
# ============================================================================

class EvolvableAgent(Agent):
    def __init__(self, genes=None, name="Evolved Agent"):
        if genes is None:
            genes = [random.random() for _ in range(6)]
        self.genes = genes
        super().__init__(name, f"Genes: [{', '.join([f'{g:.2f}' for g in genes])}]")

    def choose_action(self) -> bool:
        if self.round_num < 3:
            return random.random() < self.genes[0]
        memory_length = max(1, int(self.genes[4] * 10) + 1)
        recent_history = self.history[-memory_length:]
        cooperation_rate = sum(recent_history) / len(recent_history)
        if cooperation_rate >= 0.5:
            return random.random() < self.genes[1]
        if random.random() < self.genes[2]:
            return UNDERCUT
        elif random.random() < self.genes[3]:
            return INVEST
        return UNDERCUT


In [ ]:
# ============================================================================
# PART 2: DEFINE THE OPPONENT ECOSYSTEM
# ============================================================================
full_opponent_pool = [
    AlwaysInvestAgent(), TitForTatAgent(), GrimTriggerAgent(), PavlovAgent(),
    TitForTwoTatsAgent(), GenerousTitForTatAgent(), AdaptiveAgent(),
    AlwaysUndercutAgent(), SuspiciousTitForTatAgent(), GradualAgent(),
    ProberAgent(), HardMajorityAgent(), SoftMajorityAgent(),
    RandomAgent(0.9), RandomAgent(0.7), RandomAgent(0.5),
    RandomAgent(0.3), RandomAgent(0.1)
]
opponent_pool = random.sample(full_opponent_pool, 10)
print('Training opponents:', len(opponent_pool))


In [ ]:
# ============================================================================
# PART 3: GENETIC ALGORITHM FUNCTIONS — COMPLETED
# ============================================================================
def initialize_population(pop_size):
    return [EvolvableAgent(name=f"Agent_{i}") for i in range(pop_size)]

def evaluate_fitness(agent, opponents, num_rounds=50):
    total_score = 0
    for opponent in opponents:
        score, _ = Game(agent, opponent, num_rounds=num_rounds).play()
        total_score += score
    return total_score / len(opponents)

def tournament_selection(population, fitnesses, tournament_size=3):
    selected = []
    for _ in range(len(population)):
        contestants_idx = random.sample(range(len(population)), tournament_size)
        best_idx = max(contestants_idx, key=lambda i: fitnesses[i])
        selected.append(EvolvableAgent(genes=population[best_idx].genes.copy()))
    return selected

def crossover_and_mutate(parent1, parent2, mutation_rate=0.15):
    split = random.randint(1, len(parent1.genes)-1)
    child_genes = parent1.genes[:split] + parent2.genes[split:]
    for i in range(len(child_genes)):
        if random.random() < mutation_rate:
            child_genes[i] = max(0.0, min(1.0, child_genes[i] + random.gauss(0, 0.2)))
    return EvolvableAgent(genes=child_genes)

def evolve(generations=100, pop_size=30, opponents=None, mutation_rate=0.1, elitism=2):
    if opponents is None:
        opponents = opponent_pool
    population = initialize_population(pop_size)
    best_overall_agent = None
    best_overall_fitness = -float('inf')
    for gen in range(generations):
        fitnesses = [evaluate_fitness(agent, opponents) for agent in population]
        best_idx = fitnesses.index(max(fitnesses))
        if fitnesses[best_idx] > best_overall_fitness:
            best_overall_fitness = fitnesses[best_idx]
            best_overall_agent = EvolvableAgent(genes=population[best_idx].genes.copy(), name='Best Evolved Agent')
        sorted_indices = sorted(range(len(fitnesses)), key=lambda i: fitnesses[i], reverse=True)
        next_population = [EvolvableAgent(genes=population[i].genes.copy()) for i in sorted_indices[:elitism]]
        selected = tournament_selection(population, fitnesses)
        while len(next_population) < pop_size:
            p1, p2 = random.sample(selected, 2)
            next_population.append(crossover_and_mutate(p1, p2, mutation_rate))
        population = next_population
    best_overall_agent.fitness = best_overall_fitness
    return best_overall_agent


In [ ]:
# ============================================================================
# PART 4: RUN EVOLUTION
# ============================================================================
random.seed(0)
my_evolved_agent = evolve(100, 30, opponent_pool, 0.1, 2)
my_evolved_agent.name = 'Original Evolved Agent'
print('Original genes:', my_evolved_agent.genes)


In [ ]:
# ============================================================================
# PART 5: TEST THE ORIGINAL AGENT
# ============================================================================
for opponent in opponent_pool:
    my_score, opp_score = Game(my_evolved_agent, opponent, 100).play()
    print(opponent.name, my_score, opp_score)


In [ ]:
# ============================================================================
# PART 6: IMPROVE THE AGENT'S ACTION-TAKING STRATEGY
# ============================================================================
class ImprovedEvolvableAgent(Agent):
    def __init__(self, genes=None, name='Improved Agent'):
        if genes is None:
            genes = [random.random() for _ in range(6)]
        self.genes = genes
        super().__init__(name, f"Genes: {self.genes}")

    def choose_action(self) -> bool:
        if self.round_num < 3:
            return random.random() < self.genes[0]
        memory_length = max(1, int(self.genes[4] * 10) + 1)
        recent_history = self.history[-memory_length:]
        cooperation_rate = sum(recent_history) / len(recent_history)
        if cooperation_rate > 0.8:
            return random.random() < self.genes[1]
        elif cooperation_rate > 0.5:
            coop_prob = self.genes[1] * (cooperation_rate - 0.5) * 2
            return random.random() < coop_prob
        else:
            if random.random() < self.genes[3] * 0.3:
                return INVEST
            return UNDERCUT


In [ ]:
# ============================================================================
# PART 7: EVOLVE THE IMPROVED AGENT
# ============================================================================
def evolve_improved(generations=100, pop_size=30, opponents=None, mutation_rate=0.1, elitism=2):
    if opponents is None:
        opponents = opponent_pool
    population = [ImprovedEvolvableAgent(name=f"Agent_{i}") for i in range(pop_size)]
    best_overall_agent = None
    best_overall_fitness = -float('inf')
    for gen in range(generations):
        fitnesses = [evaluate_fitness(agent, opponents) for agent in population]
        best_idx = fitnesses.index(max(fitnesses))
        if fitnesses[best_idx] > best_overall_fitness:
            best_overall_fitness = fitnesses[best_idx]
            best_overall_agent = ImprovedEvolvableAgent(genes=population[best_idx].genes.copy())
        sorted_indices = sorted(range(len(fitnesses)), key=lambda i: fitnesses[i], reverse=True)
        next_population = [ImprovedEvolvableAgent(genes=population[i].genes.copy()) for i in sorted_indices[:elitism]]
        selected = tournament_selection(population, fitnesses)
        while len(next_population) < pop_size:
            p1, p2 = random.sample(selected, 2)
            child = crossover_and_mutate(p1, p2, mutation_rate)
            next_population.append(ImprovedEvolvableAgent(genes=child.genes))
        population = next_population
    best_overall_agent.fitness = best_overall_fitness
    return best_overall_agent

random.seed(0)
my_improved_agent = evolve_improved(100, 30, opponent_pool, 0.1, 2)
my_improved_agent.name = 'Cyber Sentinel'
print('Improved genes:', my_improved_agent.genes)


In [ ]:
# ============================================================================
# PART 8.1: COMPARE ORIGINAL VS. IMPROVED
# ============================================================================
for opponent in full_opponent_pool:
    my_score, opp_score = Game(my_improved_agent, opponent, 100).play()
    print(opponent.name, my_score, opp_score)


In [ ]:
# ============================================================================
# PART 8.2: FINAL TOURNAMENT LEADERBOARD
# ============================================================================
tournament = Tournament(full_opponent_pool + [my_improved_agent], rounds_per_match=100, num_tournaments=1)
tournament.run_tournament()
rankings = tournament.get_rankings()
for rank, (name, score) in enumerate(rankings, 1):
    print(rank, name, score)


# Unit 5 Part 2: Benchmark Analysis — Completed

## Performance Summary

| Opponent | Result | My Score | Opp. Score |
|---|---:|---:|---:|
| Always Invest | W | 354 | 219 |
| Always Undercut | L | 97 | 112 |
| Tit-for-Tat | T | 269 | 269 |
| Grim Trigger | L | 118 | 133 |
| Pavlov | W | 267 | 112 |
| Tit-for-Two-Tats | W | 336 | 221 |
| Generous Tit-for-Tat | W | 282 | 272 |
| Adaptive | W | 282 | 252 |
| Random (0.7) | L | 234 | 294 |
| Random (0.3) | W | 201 | 116 |
| **Total** | **6 W / 3 L / 1 T** |  |  |

### 1. What was your agent's worst loss and why?
The largest loss in this benchmark set was against Random (0.7), 234–294, a 60-point differential. Its inconsistent but fairly cooperative behavior made it harder for the original evolved agent to classify reliably, so the agent sometimes cooperated at costly moments. The losses to Always Undercut and Grim Trigger were smaller, showing that predictable aggression was easier to contain than mixed behavior.

### 2. What pattern do you see in wins vs. losses?
The agent generally did well against cooperative, majority-based, and moderately random strategies, but it struggled with persistent aggression and one less predictable high-cooperation random strategy. This suggests the strategy can exploit stable patterns but is more vulnerable when the opponent's behavior either locks into punishment or changes unpredictably.

### 3. What would make an agent more competitive?
I would make the action logic more selective about when to cooperate and quicker to respond when recent cooperation drops. A higher cooperation threshold, stronger retaliation, and reduced forgiveness should make the agent harder to exploit while still allowing it to earn strong mutual-cooperation payoffs against trustworthy opponents.


# Post-Evolution Reflection Questions — Completed

### 1. Generalization Analysis
The improved agent was evolved against 10 training opponents and then tested in the full 19-agent tournament. It generalized well enough to finish 3rd overall, suggesting that training against a varied sample produced behavior that transferred beyond the training set, although unseen strategies can still expose weaknesses.

### 2. Tournament Performance
Cyber Sentinel finished **3rd out of 19** with a total score of **4,748**. Gradual and Grim Trigger ranked above it, while Soft Majority and Generous Tit-for-Tat were immediately below it; these results show that stable retaliation and reciprocity remained strong in the broader tournament.

### 3. Strategy Analysis
The final evolved genes were **[1.0, 1.0, 0.7690, 0.0, 0.1385, 1.0]**. In the improved decision logic, Cyber Sentinel begins cooperatively, strongly rewards very cooperative opponents, becomes cautious in the middle range, and retaliates when cooperation falls below 50%; its evolved forgiveness value of 0 makes that retaliation especially firm. Genes 2 and 5 remain in the chromosome but are not directly used by this improved action rule, so the behavior is driven mainly by initial cooperation, cooperation response, forgiveness, and memory.

### 4. Trade-offs
Cyber Sentinel performs strongly against many cooperative and mixed opponents, but its worst final head-to-head differential was against Always Undercut, where it lost 97–112. For Unit 6, I would make the currently unused genes control meaningful behaviors such as the cooperation threshold and punishment duration so evolution can optimize those decisions too.

### 5. The Environment Matters
The 19-agent benchmark is still more predictable than a Unit 6 environment with 30+ student-designed strategies. Cyber Sentinel's selective cooperation and retaliation should provide some robustness, but unfamiliar probing or deceptive strategies may change its performance, so broader training diversity would reduce overfitting risk.


In [ ]:
# ============================================================================
# PART 9: EXPORT COMPLETE AGENT AS PYTHON MODULE
# ============================================================================
STUDENT_NAME = "Elizabeth Ineintiemo"
my_improved_agent.name = "Cyber Sentinel"
my_improved_agent.description = (
    "Selective cooperator that rewards highly cooperative opponents "
    "and retaliates strongly against repeated undercutting."
)

export_filename = "elizabethineintiemo_tournament_agent.py"
genes_literal = repr([float(g) for g in my_improved_agent.genes])
module_code = f'''from agents import Agent, INVEST, UNDERCUT
import random

class ElizabethIneintiemoAgent(Agent):
    def __init__(self):
        self.genes = {genes_literal}
        self.student_name = "Elizabeth Ineintiemo"
        super().__init__(
            name="Cyber Sentinel",
            description="Selective cooperator that rewards highly cooperative opponents and retaliates strongly against repeated undercutting."
        )

    def choose_action(self) -> bool:
        if self.round_num < 3:
            return random.random() < self.genes[0]
        memory_length = max(1, int(self.genes[4] * 10) + 1)
        recent_history = self.history[-memory_length:]
        cooperation_rate = sum(recent_history) / len(recent_history)
        if cooperation_rate > 0.8:
            return random.random() < self.genes[1]
        elif cooperation_rate > 0.5:
            coop_prob = self.genes[1] * (cooperation_rate - 0.5) * 2
            return random.random() < coop_prob
        else:
            if random.random() < self.genes[3] * 0.3:
                return INVEST
            return UNDERCUT

def get_agent():
    return ElizabethIneintiemoAgent()
'''
Path(export_filename).write_text(module_code, encoding="utf-8")
print("ASSIGNMENT EXPORT COMPLETE")
print("Student:", STUDENT_NAME)
print("Agent:", my_improved_agent.name)
print("Genes:", my_improved_agent.genes)
print("Tournament file:", export_filename)
